# Практическое занятие № 12 — оценка качества, ROC/PR и цена ошибок
**ТюмГУ · Технологии искусственного интеллекта — 1 · 90 минут**  
**Цель:** сопоставить классификаторы по пороговым и ранжирующим метрикам, выбрать модель и порог по validation, оценить влияние стоимости ошибок и дисбаланса классов. 

**Данные:** тот же `orders_dirty.csv` и `split_ids_06.csv` при наличии. `test` не используем до последнего шага; если он уже изучался на занятии № 10, итоговая оценка **не является независимой** — для реальной системы требуется свежая будущая выборка.

## 1. Проверка данных и точности протокола (0–14 мин)
Воспроизводим тот же train/validation/test, конвейер признаков и набор моделей занятия № 11. Не изменяйте очистку и разбиение после анализа качества.

In [1]:
# Общая подготовка для практик 11 и 12
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PATH = Path('orders_dirty.csv')
if not PATH.exists():
    raise FileNotFoundError('Загрузите orders_dirty.csv в файлы Colab или измените PATH.')

df_raw = pd.read_csv(PATH, encoding='utf-8-sig')
expected = {'order_id','created_at','distance_km','zone','queue_size',
            'weather','promised_minutes','actual_minutes','late'}
missing_cols = expected - set(df_raw.columns)
assert not missing_cols, f'Отсутствуют необходимые поля: {sorted(missing_cols)}'
print('Сырой набор:', df_raw.shape, '; полные дубликаты:', int(df_raw.duplicated().sum()))
if 'courier_load' not in df_raw.columns:
    # Поле есть в методичке, но отсутствует в фактическом источнике практик 5–10.
    # Не восстанавливаем значения по целевой переменной или другим косвенным данным.
    df_raw['courier_load'] = np.nan
    print('courier_load отсутствует в источнике: все значения NaN; признак неинформативен.')

# Удаление полных повторов: не обучаем статистику и не используем test.
df = df_raw.drop_duplicates().copy()
df['late'] = pd.to_numeric(df['late'], errors='coerce')
actual = pd.to_numeric(df['actual_minutes'], errors='coerce')
promised = pd.to_numeric(df['promised_minutes'], errors='coerce')
valid_rule = actual.notna() & promised.notna() & df['late'].isin([0, 1])
conflict = valid_rule & (df['late'] != ((actual - promised) > 15).astype(int))
print('Конфликтных меток (проверяемых):', int(conflict.sum()))
df = df.loc[~conflict & df['late'].isin([0, 1])].copy()
df['late'] = df['late'].astype(int)
# Неполные / конфликтные сведения об идентификаторе требуют отдельного решения.
assert df['order_id'].notna().all(), 'Обнаружены пустые ID заказов.'
assert not df['order_id'].duplicated().any(), 'Есть повторные ID: требуется анализ причины, не случайное удаление.'
print('После удаления полных повторов:', len(df_raw.drop_duplicates()))
print('Рабочая копия:', df.shape)
print('Исключено строк с недопустимой меткой:', int((~df_raw.drop_duplicates()['late'].isin([0, 1])).sum()))

split_path = Path('split_ids_06.csv')  # Необязательный экспорт из практики № 6
if split_path.exists():
    splits = pd.read_csv(split_path, dtype={'order_id': str})
    assert {'order_id', 'split'} <= set(splits.columns)
    assert set(splits['split']) <= {'train', 'val', 'test'}
    assert splits['order_id'].is_unique
    x = df.assign(order_id=df['order_id'].astype(str)).merge(
        splits[['order_id', 'split']], on='order_id', how='left', validate='one_to_one')
    assert x['split'].notna().all(), 'В split_ids_06.csv перечислены не все рабочие заказы.'
    assert set(splits['order_id']) == set(x['order_id'])
    train = x.loc[x['split'] == 'train'].drop(columns='split').copy()
    val   = x.loc[x['split'] == 'val'].drop(columns='split').copy()
    test  = x.loc[x['split'] == 'test'].drop(columns='split').copy()
else:
    # Воссоздание схемы практики № 6, если файл с IDs не был сохранён.
    train_dev, test = train_test_split(df, test_size=.20, stratify=df['late'], random_state=42)
    train, val = train_test_split(train_dev, test_size=.25,
                                 stratify=train_dev['late'], random_state=42)
    split_export = pd.concat([
       train[['order_id']].assign(split='train'),
       val[['order_id']].assign(split='val'),
       test[['order_id']].assign(split='test')], ignore_index=True)
    split_export.to_csv('split_ids_06.csv', index=False)

sets = [set(a['order_id']) for a in (train, val, test)]
assert len(sets[0]&sets[1]) == len(sets[0]&sets[2]) == len(sets[1]&sets[2]) == 0
assert sets[0]|sets[1]|sets[2] == set(df['order_id'])
assert len(train)+len(val)+len(test) == len(df)
print('train / val / test:', len(train), len(val), len(test))
print('Доли от рабочего N:', [round(len(a)/len(df), 4) for a in (train, val, test)])
print('late=1 на train и validation:', round(train['late'].mean(), 4), round(val['late'].mean(), 4))
print('Утечка фактической длительности недопустима: actual_minutes становится известна после доставки.')

Сырой набор: (600, 10) ; полные дубликаты: 1
courier_load отсутствует в источнике: все значения NaN; признак неинформативен.
Конфликтных меток (проверяемых): 1
После удаления полных повторов: 599
Рабочая копия: (598, 11)
Исключено строк с недопустимой меткой: 0
train / val / test: 358 120 120
Доли от рабочего N: [0.5987, 0.2007, 0.2007]
late=1 на train и validation: 0.4693 0.4667
Утечка фактической длительности недопустима: actual_minutes становится известна после доставки.


In [2]:
# В Colab функция экспортируется в обычный модуль для безопасного joblib.dump.
# При перезапуске блокнота модуль восстанавливается автоматически.
from pathlib import Path
Path('tyumgu_feature_rules.py').write_text("# Правила преобразования признаков для занятий 7–12\nimport numpy as np\nimport pandas as pd\nNUM_BASE = ['distance_km', 'queue_size', 'courier_load', 'promised_minutes']\nCAT = ['zone', 'weather']\nNUM = NUM_BASE + ['hour_sin', 'hour_cos', 'planned_speed_kmh']\nINPUT = NUM_BASE + CAT + ['created_at']\n\ndef make_features(X):\n    f = X.loc[:, INPUT].copy()\n    for name in NUM_BASE:\n        f[name] = pd.to_numeric(f[name], errors='coerce')\n    for name in NUM_BASE:\n        f.loc[f[name] < 0, name] = np.nan\n    f.loc[f['promised_minutes'] == 0, 'promised_minutes'] = np.nan\n    for name in CAT:\n        s = f[name].astype('string').str.strip()\n        s = s.mask(s.str.lower().isin(['', 'nan', 'none', 'нет данных', 'неизвестно', 'null']))\n        f[name] = s.astype(object).where(s.notna(), np.nan)\n    time = pd.to_datetime(f['created_at'], errors='coerce')\n    hour = time.dt.hour.astype(float)\n    f['hour_sin'] = np.sin(2*np.pi*hour/24)\n    f['hour_cos'] = np.cos(2*np.pi*hour/24)\n    f['planned_speed_kmh'] = 60*f['distance_km']/f['promised_minutes']\n    f['planned_speed_kmh'] = f['planned_speed_kmh'].replace([np.inf, -np.inf], np.nan)\n    return f.drop(columns='created_at')\n", encoding='utf-8')
from tyumgu_feature_rules import NUM_BASE, NUM, CAT, INPUT, make_features
F_train, F_val = [make_features(part) for part in (train, val)]
assert set(F_train.columns) == set(NUM + CAT)
assert all(col not in F_train.columns for col in ('late','actual_minutes','order_id'))
print('Матрица до кодирования:', F_train.shape)
print('Числовые столбцы:', NUM, '; категориальные:', CAT)

Матрица до кодирования: (358, 9)
Числовые столбцы: ['distance_km', 'queue_size', 'courier_load', 'promised_minutes', 'hour_sin', 'hour_cos', 'planned_speed_kmh'] ; категориальные: ['zone', 'weather']


In [3]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

numeric_pipe = Pipeline([
    ('fill', SimpleImputer(strategy='median', add_indicator=True,
                           keep_empty_features=True)),
    ('scale', StandardScaler())
])
cat_pipe = Pipeline([
    ('fill', SimpleImputer(strategy='constant', fill_value='неизвестно')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocess = ColumnTransformer([
    ('num', numeric_pipe, NUM),
    ('cat', cat_pipe, CAT)
], remainder='drop', verbose_feature_names_out=True)

def build_pipeline(classifier):
    return Pipeline([
        ('derive', FunctionTransformer(make_features, validate=False)),
        ('prep', clone(preprocess)),
        ('clf', classifier)
    ])

Xtrain = train[INPUT].copy(); Xval = val[INPUT].copy()
ytrain = train['late'].astype(int).copy()
yval = val['late'].astype(int).copy()
# test хранится отдельно и не используется для подбора модели.
assert ytrain.nunique() == 2, 'Для обучения необходимы 0 и 1 в train.'
assert {'order_id','late','actual_minutes'}.isdisjoint(Xtrain.columns)
assert set(train['order_id']).isdisjoint(val['order_id'])
assert set(train['order_id']).isdisjoint(test['order_id'])
assert set(val['order_id']).isdisjoint(test['order_id'])
print('Обучение:', Xtrain.shape, '| validation:', Xval.shape,
      '| отложенный test:', len(test))
probe = build_pipeline(DummyClassifier(strategy='prior')).named_steps['prep']
probe.fit(F_train, ytrain)
print('Признаков до/после кодирования:', F_train.shape[1], len(probe.get_feature_names_out()))
assert len(probe.get_feature_names_out()) > F_train.shape[1]
print('Доли класса late=1 (train / validation):',
      [round(v.mean(), 3) for v in (ytrain,yval)])

Обучение: (358, 7) | validation: (120, 7) | отложенный test: 120
Признаков до/после кодирования: 9 18
Доли класса late=1 (train / validation): [np.float64(0.469), np.float64(0.467)]


## 2. Обучить кандидатов только на train (14–29 мин)
Используйте модели и стартовые параметры занятия 11. `fit` допустим только для train.

In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

# Одинаковый препроцессинг и те же train / validation для честного сравнения.
# Названия и параметры фиксированы ДО анализа test.
CANDIDATES = {
    'Dummy': DummyClassifier(strategy='prior'),
    'LogReg': LogisticRegression(C=1, max_iter=3000, random_state=42),
    'Tree': DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42),
    'Forest': RandomForestClassifier(n_estimators=120, max_depth=6,
                                      min_samples_leaf=3, random_state=42,
                                      n_jobs=-1),
    'Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05,
                                            max_depth=2, min_samples_leaf=3,
                                            random_state=42)
}

models = {}
for name, estimator in CANDIDATES.items():
    model = build_pipeline(estimator)
    model.fit(Xtrain, ytrain)  # Только train!
    models[name] = model
    assert len(model.predict(Xval)) == len(yval)
print('Обучены:', ', '.join(models))

Обучены: Dummy, LogReg, Tree, Forest, Boosting


In [5]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    RocCurveDisplay, PrecisionRecallDisplay
)
import matplotlib.pyplot as plt

def positives(model, X):
    # При чтении вероятностей проверяем позицию именно класса 1.
    classes = model.named_steps['clf'].classes_
    ix = int(np.flatnonzero(classes == 1)[0])
    return model.predict_proba(X)[:, ix]

def metrics_at(y, p, threshold=0.50, model_name=''):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    d = {
        'model': model_name, 'threshold': float(threshold),
        'N': len(y), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'accuracy': accuracy_score(y, pred),
        'balanced_accuracy': balanced_accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'cost_rub': 200 * int(fp) + 2000 * int(fn)
    }
    if len(np.unique(y)) == 2:
        d['roc_auc'] = roc_auc_score(y, p)
        d['average_precision'] = average_precision_score(y, p)
    else:
        d['roc_auc'] = float('nan')
        d['average_precision'] = float('nan')
    return d

## 3. Порог 0.50: матрица ошибок и метрики (29–43 мин)
Матрица `[[TN,FP],[FN,TP]]` при `labels=[0,1]`, строки — истинные метки. Посчитайте Accuracy, Precision, Recall, F1, Balanced Accuracy. Обязательно покажите численность положительного класса; 0 при делении на ноль означает техническую договорённость `zero_division=0`.

In [6]:
preds_val={name:positives(m,Xval) for name,m in models.items()}
base_results = pd.DataFrame([metrics_at(yval, p, .50, name)
                             for name,p in preds_val.items()])
display(base_results.sort_values('average_precision',ascending=False).round(3))
base_results.to_csv('12_validation_at_050.csv',index=False)
print('Доля задержек на validation:',round(yval.mean(),4),'; n=',len(yval))
print('Классы на validation: вовремя =',int((yval==0).sum()),
      ', задержка =',int((yval==1).sum()))
for name in CANDIDATES:
    r=base_results.set_index('model').loc[name]
    print(f"{name}: [[TN={int(r.TN)}, FP={int(r.FP)}], [FN={int(r.FN)}, TP={int(r.TP)}]]")
print('Для Forest: TN — заказ вовремя и тревоги нет; FP — ложная тревога (200 ₽);')
print('FN — пропущенная задержка (2000 ₽); TP — задержка распознана заранее.')
assert all(base_results['TN']+base_results['FP']+base_results['FN']+base_results['TP']==len(yval))

,model,threshold,N,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,f1,cost_rub,roc_auc,average_precision
3,Forest,0.5,120,42,22,17,39,0.675,0.676,0.639,0.696,0.667,38400,0.744,0.763
1,LogReg,0.5,120,49,15,19,37,0.717,0.713,0.712,0.661,0.685,41000,0.755,0.741
4,Boosting,0.5,120,42,22,22,34,0.633,0.632,0.607,0.607,0.607,48400,0.680,0.693
2,Tree,0.5,120,43,21,22,34,0.642,0.640,0.618,0.607,0.613,48200,0.654,0.626
0,Dummy,0.5,120,64,0,56,0,0.533,0.500,0.000,0.000,0.000,112000,0.500,0.467


Доля задержек на validation: 0.4667 ; n= 120
Классы на validation: вовремя = 64 , задержка = 56
Dummy: [[TN=64, FP=0], [FN=56, TP=0]]
LogReg: [[TN=49, FP=15], [FN=19, TP=37]]
Tree: [[TN=43, FP=21], [FN=22, TP=34]]
Forest: [[TN=42, FP=22], [FN=17, TP=39]]
Boosting: [[TN=42, FP=22], [FN=22, TP=34]]
Для Forest: TN — заказ вовремя и тревоги нет; FP — ложная тревога (200 ₽);
FN — пропущенная задержка (2000 ₽); TP — задержка распознана заранее.


## 4. ROC-AUC и PR: сравнение ранжирования (43–58 мин)
AUC оценивает ранжирование при разных порогах. AP — взвешенная средняя Precision по приращениям Recall. При сильном дисбалансе ROC-AUC и AP могут приводить к разным выводам. Проведите сравнение не менее трёх обучаемых моделей.

In [7]:
assert yval.nunique()==2, ('Нужны оба класса на validation для ROC-AUC; '
                           'пересмотрите допустимость разбиения на малом наборе.')
fig,axes=plt.subplots(1,2,figsize=(11.2,4.4))
for name in ['LogReg','Tree','Forest','Boosting']:
    p=preds_val[name]
    fpr,tpr,_=roc_curve(yval,p)
    prec,rec,_=precision_recall_curve(yval,p)
    axes[0].plot(fpr,tpr,label=f'{name}: AUC={roc_auc_score(yval,p):.2f}')
    axes[1].plot(rec,prec,label=f'{name}: AP={average_precision_score(yval,p):.2f}')
axes[0].plot([0,1],[0,1],'--',color='gray',linewidth=1)
axes[1].axhline(yval.mean(),linestyle='--',color='gray',label='Доля класса 1')
axes[0].set(xlabel='FPR',ylabel='TPR (Recall)',title='ROC · validation')
axes[1].set(xlabel='Recall',ylabel='Precision',title='PR · validation')
for ax in axes: ax.legend(fontsize=7);ax.grid(alpha=.2)
plt.tight_layout();plt.savefig('12_roc_pr.png',dpi=170);plt.show()

C:\Users\VladS\git\utmn-master-degree\tii\lb12\tmp\ipykernel_15992\941579214.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.savefig('12_roc_pr.png',dpi=170);plt.show()


## 5. Подбор модели и порога только по validation (58–75 мин)
Затраты: **FP = 200 руб., FN = 2000 руб.**. Для каждой обучаемой модели посчитайте стоимость на сетке порогов `0.05, 0.10, ..., 0.95` и добавьте теоретический байесовский порог `200/(200+2000)` (применим при корректных вероятностях и заданных затратах). 

**Правило выбора, заданное заранее:** сначала минимальная стоимость; при равенстве — максимальный Recall; затем максимальная AP; затем порядок моделей `LogReg → Tree → Forest → Boosting` и близость порога к 0.5. AP не зависит от порога и используется только как дополнительный критерий. На практике множественный перебор может приводить к оптимизму на маленькой validation.

In [8]:
C_FP=200;C_FN=2000
thresholds=sorted(set([round(t,2) for t in np.arange(.05,1.,.05)] + [C_FP/(C_FP+C_FN)]))
priority={'LogReg':0,'Tree':1,'Forest':2,'Boosting':3}
records=[]
for name in priority:
    for t in thresholds:
        rec=metrics_at(yval,preds_val[name],t,name)
        rec['model_priority']=priority[name]
        rec['distance_to_half']=abs(t-0.5)
        records.append(rec)
grid=pd.DataFrame(records)
# Tie-breaking deterministic and visible in the result.
ranking=grid.sort_values(['cost_rub','recall','average_precision','model_priority','distance_to_half','threshold'],
                         ascending=[True,False,False,True,True,True])
chosen=ranking.iloc[0]
selected_name=str(chosen['model']); selected_t=float(chosen['threshold'])
print('Выбраны:',selected_name,'порог=',round(selected_t,4),'C(val)=',int(chosen['cost_rub']))
print('Теоретический порог (только при достоверной калибровке):',round(C_FP/(C_FP+C_FN),4))
display(ranking.head(10)[['model','threshold','FP','FN','recall','f1','cost_rub']])
grid.to_csv('12_validation_threshold_grid.csv',index=False)
assert selected_name in priority
assert selected_t in thresholds
summary=grid.sort_values(['cost_rub','recall','average_precision','distance_to_half','threshold'],
                         ascending=[True,False,False,True,True]).groupby('model',as_index=False).first()
print('Лучшее валидaционное решение для каждой модели:')
display(summary[['model','threshold','FP','FN','recall','average_precision','cost_rub']]
        .sort_values('cost_rub').round(4))
print('Проверено комбинаций модель × порог:',len(grid))


Выбраны: LogReg порог= 0.15 C(val)= 12200
Теоретический порог (только при достоверной калибровке): 0.0909


,model,threshold,FP,FN,recall,f1,cost_rub
3,LogReg,0.150000,61,0,1.0,0.647399,12200
44,Forest,0.200000,63,0,1.0,0.640000,12600
43,Forest,0.150000,63,0,1.0,0.640000,12600
2,LogReg,0.100000,63,0,1.0,0.640000,12600
1,LogReg,0.090909,63,0,1.0,0.640000,12600
0,LogReg,0.050000,63,0,1.0,0.640000,12600
63,Boosting,0.150000,63,0,1.0,0.640000,12600
42,Forest,0.100000,64,0,1.0,0.636364,12800
41,Forest,0.090909,64,0,1.0,0.636364,12800
40,Forest,0.050000,64,0,1.0,0.636364,12800


Лучшее валидaционное решение для каждой модели:


,model,threshold,FP,FN,recall,average_precision,cost_rub
2,LogReg,0.15,61,0,1.0,0.7406,12200
0,Boosting,0.15,63,0,1.0,0.6929,12600
1,Forest,0.20,63,0,1.0,0.7625,12600
3,Tree,0.15,64,0,1.0,0.6259,12800


Проверено комбинаций модель × порог: 80


In [9]:
fig,ax=plt.subplots(figsize=(8,4.3))
for name in priority:
    sub=grid.loc[grid['model']==name].sort_values('threshold')
    ax.plot(sub['threshold'],sub['cost_rub'],label=name,marker='.',markersize=3)
ax.scatter([selected_t],[chosen['cost_rub']],s=80,zorder=10,label='Выбранное решение')
ax.set(xlabel='Порог t',ylabel='Учебная стоимость FP/FN, ₽',
       title='Стоимость ошибок на validation (не на test)')
ax.grid(alpha=.2);ax.legend();plt.tight_layout()
plt.savefig('12_cost_thresholds.png',dpi=160);plt.show()

C:\Users\VladS\git\utmn-master-degree\tii\lb12\tmp\ipykernel_15992\4029329335.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig('12_cost_thresholds.png',dpi=160);plt.show()


## 6. Проверить дисбаланс и устойчивость результатов (75–81 мин)
Сравните `accuracy` и `balanced_accuracy`, AP и долю положительного класса. Объясните, почему высокая Accuracy для Dummy не обязательно означает пригодность. Покажите изменение размера validation и оцените чувствительность результатов к 1–2 ошибкам в таблице. Дополнительно можно выполнить временной сплит как отдельный эксперимент **без подбора по test**.

In [10]:
print('Размеры:',{'train':len(ytrain),'val':len(yval), 'test_НЕ_ОЦЕНЁН':len(test)})
print('Доля задержек:', {'train':round(float(ytrain.mean()),3),
                         'val':round(float(yval.mean()),3)})
print('Сравнение валидационных метрик при 0.5:')
display(base_results[['model','accuracy','balanced_accuracy','recall',
                      'roc_auc','average_precision','cost_rub']].round(3))
print('Dummy при пороге 0.5: accuracy =',round(float(base_results.set_index('model').loc['Dummy','accuracy']),3),
      ', balanced_accuracy =',round(float(base_results.set_index('model').loc['Dummy','balanced_accuracy']),3),
      ', recall = 0: все 56 задержек пропущены.')
print('Одна дополнительная FN увеличивает стоимость на 2000 ₽; две — на 4000 ₽.')
print('На validation 120 заказов: один случай — 0.83 процентного пункта,')
print('поэтому перебор',len(grid),'комбинаций может переобучить выбор на эту выборку.')
print('Тот же test уже анализировался в практике 10; итоговая оценка имеет учебный, не независимый статус.')


Размеры: {'train': 358, 'val': 120, 'test_НЕ_ОЦЕНЁН': 120}
Доля задержек: {'train': 0.469, 'val': 0.467}
Сравнение валидационных метрик при 0.5:


,model,accuracy,balanced_accuracy,recall,roc_auc,average_precision,cost_rub
0,Dummy,0.533,0.500,0.000,0.500,0.467,112000
1,LogReg,0.717,0.713,0.661,0.755,0.741,41000
2,Tree,0.642,0.640,0.607,0.654,0.626,48200
3,Forest,0.675,0.676,0.696,0.744,0.763,38400
4,Boosting,0.633,0.632,0.607,0.680,0.693,48400


Dummy при пороге 0.5: accuracy = 0.533 , balanced_accuracy = 0.5 , recall = 0: все 56 задержек пропущены.
Одна дополнительная FN увеличивает стоимость на 2000 ₽; две — на 4000 ₽.
На validation 120 заказов: один случай — 0.83 процентного пункта,
поэтому перебор 80 комбинаций может переобучить выбор на эту выборку.
Тот же test уже анализировался в практике 10; итоговая оценка имеет учебный, не независимый статус.


## 7. Заморозить настройки и однократно оценить test (81–88 мин)
**Важно:** не переобучайте модель, не изменяйте гиперпараметры, не меняйте порог после этого шага. В журнале отчёта укажите, смотрели ли вы на этот же test в практиках № 9–10. Если да, результат учебный, а не независимая оценка обобщения. Итоговый test-файл не используйте повторно для выбора решений.

In [11]:
# Эта ячейка — ЕДИНСТВЕННОЕ место оценки качества на test в данной практике.
# Раскрываем отложенный test только после фиксации выбора по validation.
Xtest = test[INPUT].copy()
ytest = test['late'].astype(int).copy()
assert {'order_id','late','actual_minutes'}.isdisjoint(Xtest.columns)
frozen_model=models[selected_name]
test_probability=positives(frozen_model,Xtest)
final_record=metrics_at(ytest,test_probability,selected_t,selected_name)
test_eval=pd.DataFrame([final_record])
display(test_eval.round(4))
test_eval.to_csv('12_final_test_once.csv',index=False)
from pathlib import Path
import json
config={'model':selected_name,'threshold':selected_t,
        'validation_cost_rub':int(chosen['cost_rub']),
        'C_FP_rub':C_FP,'C_FN_rub':C_FN,
        'test_status':'учебная контрольная оценка; тот же test уже анализировался в практике 10'}
Path('12_selected_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),
                                           encoding='utf-8')
cm=confusion_matrix(ytest,(test_probability>=selected_t).astype(int),labels=[0,1])
fig,ax=plt.subplots(figsize=(4.3,3.7)); im=ax.imshow(cm,cmap='Blues')
for (i,j),v in np.ndenumerate(cm):
    ax.text(j,i,str(int(v)),va='center',ha='center',fontsize=13)
ax.set_xticks([0,1],['0 вовремя','1 задержка'])
ax.set_yticks([0,1],['0 вовремя','1 задержка'])
ax.set(xlabel='Прогноз',ylabel='Факт',title='Матрица ошибок · test')
plt.colorbar(im,ax=ax,fraction=.046);plt.tight_layout();plt.savefig('12_test_confusion_matrix.png',dpi=160);plt.show()
print('Контрольный test выполнен один раз; статус: учебная оценка, не независимая.')


,model,threshold,N,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,f1,cost_rub,roc_auc,average_precision
0,LogReg,0.15,120,5,59,1,55,0.5,0.5301,0.4825,0.9821,0.6471,13800,0.7458,0.7353


Контрольный test выполнен один раз; статус: учебная оценка, не независимая.


C:\Users\VladS\git\utmn-master-degree\tii\lb12\tmp\ipykernel_15992\3919495602.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.colorbar(im,ax=ax,fraction=.046);plt.tight_layout();plt.savefig('12_test_confusion_matrix.png',dpi=160);plt.show()


## 8. Интерпретация, контрольные вопросы и сдача (88–90 мин)
1. Почему Accuracy способна скрывать проблемы при редком положительном классе?
2. Чем ROC-AUC отличается от AP и почему их нельзя сравнивать как одно число «качества»?
3. Как фиксировать TN/FP/FN/TP для класса `late=1`?
4. Что означает порог `200/2200`, и почему он может быть неоптимальным у некалиброванной модели?
5. Что происходит с Precision/Recall при изменении порога?
6. Почему test с занятия 10 уже не является полностью независимым?

**Сдача:** `12_student.ipynb`, отчёт `Группа_Фамилия_З12_Отчёт.docx/pdf`, CSV таблиц, PNG ROC/PR и стоимости, `12_selected_config.json`, итоговый CSV проверки test. Выполнить «Перезапуск → Выполнить все».

**Ответы и проверка интерпретации.**
1. Accuracy может быть высокой при прогнозе только большинства; для задержек важны Recall, Precision и AP. Сравнение с Dummy и долей late=1 приведено выше.
2. ROC-AUC оценивает отделение классов по FPR/TPR при всех порогах; AP суммирует Precision по приращениям Recall и чувствительнее к доле положительного класса. Это разные шкалы качества.
3. При late=1: TN — вовремя без тревоги, FP — ложная тревога, FN — пропущенная задержка, TP — найденная задержка. FPR=FP/(FP+TN), TPR=TP/(TP+FN).
4. Порог 200/2200 ≈ 0,091 следует из постоянных затрат FP/FN и корректно калиброванных вероятностей. На малой выборке и при некалиброванной модели минимум validation может быть другим.
5. Снижение порога обычно повышает Recall и увеличивает число FP; Precision может снизиться. Стоимость учитывает ошибки первого рода (FP) и второго рода (FN).
6. Test уже просматривался в практике 10. Повторное использование делает его зависимым от прошлых решений; полученная оценка учебная. Для независимой оценки нужна новая временная выборка.
